# First look: when does each word get painted?

Stable Diffusion reads the prompt at every denoising step through 16 cross-attention layers. Here we record, for every word and every step, how much attention the word receives and where in the image it lands. Two measurements per word:

* **Attention share**: of the attention going to the prompt's own words, what fraction this word gets at each step.
* **Commit step**: the first step after which the word's heatmap stays correlated (Pearson r of at least 0.8) with its final layout. Read it as the moment the model has decided where the word goes.

The recorder replaces only the cross-attention computation and reproduces the stock output; section 2 measures how close.

A note on step counts: SD 1.5's default PNDM scheduler evaluates the UNet once more than the number of inference steps, so a 30-step run yields 31 recorded entries. The full study switches to a scheduler with exactly one evaluation per step; this first look keeps the default so the images match Milestone 1.

In [1]:
import sys, json, textwrap
from collections import Counter
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np, torch, matplotlib.pyplot as plt
from w2w import generate
from w2w.attention import (AttentionStore, install_recorder, remove_recorder, step_callback, word_tokens,
                           attention_share, commit_step, correlation_curve, final_step_drop, is_content)
from w2w.paths import RESULTS

OUT = RESULTS / "m1_word_timelines"
OUT.mkdir(parents=True, exist_ok=True)
pipe = generate.load_pipeline()
tok = pipe.tokenizer

def labelled(words):
    """(slot, word, label): a word that appears twice gets its position, e.g. 'man (1)', 'man (2)'."""
    counts, seen, out = Counter(w for _, w in words), Counter(), []
    for s, w in words:
        seen[w] += 1
        out.append((s, w, f"{w} ({seen[w]})" if counts[w] > 1 else w))
    return out

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

## 1. Prompts
The five Milestone 1 COCO captions (seed 0, 30 steps), the smoke-test prompt that came out with no cat (reproduced at its original seed 2 and 20 steps), and two stress prompts.

In [2]:
m1 = json.load(open(RESULTS / "m1_pipeline" / "summary.json"))
prompts = [{"name": f"coco_{s['image_id']}", "prompt": s["caption"], "seed": 0, "steps": 30} for s in m1["samples"]]
prompts += [
    {"name": "missing_cat", "prompt": "A cat sitting on a wooden bench in a park.", "seed": 2, "steps": 20},
    {"name": "dog_chasing_cat", "prompt": "A dog chasing a cat across a grassy yard.", "seed": 0, "steps": 30},
    {"name": "red_cube_blue_sphere", "prompt": "A red cube on top of a blue sphere.", "seed": 0, "steps": 30},
]
for p in prompts: print(p["name"], "|", p["prompt"])

coco_87470 | A bunch of cows walking on the side of the road
coco_190756 | A man flying through the air while riding a bike.
coco_234607 | A couple fighting each other over a wii remote control
coco_391722 | A man serving another man a cake with candles on it.
coco_574823 | Baseball player looking to his left while other players stand and sit around talking.
missing_cat | A cat sitting on a wooden bench in a park.
dog_chasing_cat | A dog chasing a cat across a grassy yard.
red_cube_blue_sphere | A red cube on top of a blue sphere.


## 2. Does recording change the image?
Same prompt and seed, with and without the recorder. The recorder hands the actual computation to the stock attention code and only recomputes the probabilities on the side, so the two images should be identical.

In [3]:
def run(p, record=True):
    store = AttentionStore()
    orig = install_recorder(pipe.unet, store) if record else None
    g = torch.Generator("cuda").manual_seed(p["seed"])
    try:
        out = pipe(p["prompt"], num_inference_steps=p["steps"], guidance_scale=7.5, generator=g,
                   callback_on_step_end=step_callback(store) if record else None)
    finally:
        if record: remove_recorder(pipe.unet, orig)
    return out.images[0], store

probe = prompts[-3]
img_plain, _ = run(probe, record=False)
img_rec, st = run(probe, record=True)
pix = float(np.abs(np.asarray(img_plain, np.float32) - np.asarray(img_rec, np.float32)).mean())
print(f"mean pixel difference with vs without recorder: {pix:.3f} (0 to 255 scale); steps recorded: {len(st.maps)}")
assert pix < 0.5, "the recorder changed the generated image"

mean pixel difference with vs without recorder: 0.000 (0 to 255 scale); steps recorded: 21


## 3. Record every prompt

In [4]:
records = []
for p in prompts:
    img, store = run(p)
    maps = store.stacked()                       # (UNet evaluations, 77, 64, 64)
    words = labelled(word_tokens(tok, p["prompt"]))
    slots = [s for s, _, _ in words]
    share = attention_share(maps, slots)
    sot_share = float(maps[:, 0].astype(np.float32).mean())
    commits = {lab: commit_step(maps, s) for s, w, lab in words}
    drops = {lab: final_step_drop(maps, s) for s, w, lab in words if is_content(w)}
    keep = maps[:, :max(slots) + 2]               # start token, words, end token
    np.savez_compressed(OUT / f"{p['name']}_maps.npz", maps=keep, slots=np.array(slots),
                        words=np.array([lab for _, _, lab in words]))
    img.save(OUT / f"{p['name']}.png")
    records.append({**p, "words": words, "share": share, "commits": commits, "drops": drops,
                    "img": img, "maps": maps, "sot_share": sot_share})
    print(f"{p['name']:<22} start-token share: {sot_share:.2f} | commit steps:",
          {lab: c for (s, w, lab), c in zip(words, commits.values()) if is_content(w)})

coco_87470             start-token share: 0.89 | commit steps: {'bunch': 21, 'cows': 7, 'walking': 6, 'side': 15, 'road': 5}


coco_190756            start-token share: 0.91 | commit steps: {'man': 20, 'flying': 17, 'air': 15, 'riding': 6, 'bike': 3}


coco_234607            start-token share: 0.90 | commit steps: {'couple': 18, 'fighting': 12, 'wii': 8, 'remote': 13, 'control': 14}


coco_391722            start-token share: 0.90 | commit steps: {'man (1)': 20, 'serving': 12, 'man (2)': 18, 'cake': 0, 'candles': 9}


coco_574823            start-token share: 0.91 | commit steps: {'baseball': 15, 'player': 16, 'looking': 15, 'left': 19, 'players': 17, 'stand': 15, 'sit': 13, 'talking': 18}


missing_cat            start-token share: 0.89 | commit steps: {'cat': 14, 'sitting': 14, 'wooden': 11, 'bench': 1, 'park': 11}


dog_chasing_cat        start-token share: 0.89 | commit steps: {'dog': 3, 'chasing': 9, 'cat': 3, 'across': 6, 'grassy': 12, 'yard': 20}


red_cube_blue_sphere   start-token share: 0.89 | commit steps: {'red': 13, 'cube': 7, 'top': 17, 'blue': 18, 'sphere': 3}


## 4. Is the commit step robust to its threshold?
If the order in which words commit flips when the threshold moves, the measure isn't telling us much. We check 0.7, 0.8 and 0.9.

In [5]:
sens = {}
for r in records:
    sens[r["name"]] = {thr: {lab: commit_step(r["maps"], s, threshold=thr) for s, w, lab in r["words"] if is_content(w)}
                       for thr in (0.7, 0.8, 0.9)}
    print(r["name"])
    for thr, d in sens[r["name"]].items():
        print(f"   r>={thr}: ", d)

coco_87470
   r>=0.7:  {'bunch': 19, 'cows': 5, 'walking': 4, 'side': 10, 'road': 3}
   r>=0.8:  {'bunch': 21, 'cows': 7, 'walking': 6, 'side': 15, 'road': 5}
   r>=0.9:  {'bunch': 24, 'cows': 10, 'walking': 8, 'side': 20, 'road': 9}
coco_190756
   r>=0.7:  {'man': 16, 'flying': 13, 'air': 10, 'riding': 4, 'bike': 1}
   r>=0.8:  {'man': 20, 'flying': 17, 'air': 15, 'riding': 6, 'bike': 3}
   r>=0.9:  {'man': 23, 'flying': 22, 'air': 20, 'riding': 17, 'bike': 10}
coco_234607
   r>=0.7:  {'couple': 17, 'fighting': 5, 'wii': 8, 'remote': 11, 'control': 12}
   r>=0.8:  {'couple': 18, 'fighting': 12, 'wii': 8, 'remote': 13, 'control': 14}
   r>=0.9:  {'couple': 21, 'fighting': 18, 'wii': 12, 'remote': 16, 'control': 15}
coco_391722
   r>=0.7:  {'man (1)': 17, 'serving': 10, 'man (2)': 16, 'cake': 0, 'candles': 0}
   r>=0.8:  {'man (1)': 20, 'serving': 12, 'man (2)': 18, 'cake': 0, 'candles': 9}
   r>=0.9:  {'man (1)': 23, 'serving': 19, 'man (2)': 22, 'cake': 6, 'candles': 12}


coco_574823
   r>=0.7:  {'baseball': 13, 'player': 13, 'looking': 8, 'left': 16, 'players': 13, 'stand': 13, 'sit': 9, 'talking': 15}
   r>=0.8:  {'baseball': 15, 'player': 16, 'looking': 15, 'left': 19, 'players': 17, 'stand': 15, 'sit': 13, 'talking': 18}
   r>=0.9:  {'baseball': 20, 'player': 21, 'looking': 20, 'left': 22, 'players': 21, 'stand': 17, 'sit': 15, 'talking': 22}
missing_cat
   r>=0.7:  {'cat': 14, 'sitting': 12, 'wooden': 9, 'bench': 0, 'park': 9}
   r>=0.8:  {'cat': 14, 'sitting': 14, 'wooden': 11, 'bench': 1, 'park': 11}
   r>=0.9:  {'cat': 15, 'sitting': 15, 'wooden': 13, 'bench': 6, 'park': 13}
dog_chasing_cat
   r>=0.7:  {'dog': 2, 'chasing': 6, 'cat': 2, 'across': 4, 'grassy': 8, 'yard': 17}
   r>=0.8:  {'dog': 3, 'chasing': 9, 'cat': 3, 'across': 6, 'grassy': 12, 'yard': 20}
   r>=0.9:  {'dog': 8, 'chasing': 12, 'cat': 10, 'across': 10, 'grassy': 19, 'yard': 23}
red_cube_blue_sphere
   r>=0.7:  {'red': 7, 'cube': 5, 'top': 9, 'blue': 13, 'sphere': 1}
   r>=0.8: 

## 5. Word timelines

In [6]:
def overlay(img, m):
    m = torch.from_numpy(m.astype(np.float32))[None, None]
    m = torch.nn.functional.interpolate(m, size=img.size[::-1], mode="bilinear")[0, 0].numpy()
    return (m - m.min()) / (m.max() - m.min() + 1e-8)

def timeline_figure(r, path, max_words=5):
    content = [(i, s, lab) for i, (s, w, lab) in enumerate(r["words"]) if is_content(w)][:max_words]
    steps = np.arange(r["share"].shape[0])
    fig = plt.figure(figsize=(3.6 * (2 + len(content)), 4.4))
    gs = fig.add_gridspec(1, 2 + len(content), width_ratios=[1, 1.6] + [1] * len(content))
    ax0 = fig.add_subplot(gs[0]); ax0.imshow(r["img"]); ax0.set_axis_off()
    ax0.set_title(textwrap.fill(r["prompt"], 34), fontsize=9)
    ax1 = fig.add_subplot(gs[1])
    for i, s, lab in content:
        line, = ax1.plot(steps, r["share"][:, i], lw=2, label=lab)
        c = r["commits"][lab]
        if c is not None:
            ax1.axvline(c, color=line.get_color(), ls=":", lw=1.2)
    ax1.set(xlabel="UNet evaluation (denoising step)", ylabel="share of word attention",
            title="Attention share (dotted: commit step)")
    ax1.legend(fontsize=8, frameon=False)
    n = r["maps"].shape[0]
    final = r["maps"][n - 6:n - 1].astype(np.float32).mean(0)   # same reference window as commit_step
    for k, (i, s, lab) in enumerate(content):
        ax = fig.add_subplot(gs[2 + k])
        ax.imshow(r["img"]); ax.imshow(overlay(r["img"], final[s]), cmap="inferno", alpha=0.55)
        c = r["commits"][lab]
        ax.set_title(f"'{lab}'  commit: {'never' if c is None else f'step {c}'}", fontsize=9); ax.set_axis_off()
    fig.tight_layout(); fig.subplots_adjust(top=0.82)
    fig.savefig(path, dpi=120); plt.show()

for r in records:
    timeline_figure(r, OUT / f"{r['name']}_timeline.png")

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

[figure saved under Results/m1_word_timelines/]

## 6. How a word's map settles
Correlation of each content word's heatmap with its final layout (the mean of the five evaluations before the last), step by step. The commit step is where the curve crosses 0.8 and stays above it. The final evaluation is drawn but excluded from the measure: at the lowest noise level nearly every word's map decorrelates at once, which section 7 reports as its own observation.

In [7]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5), sharey=True)
for ax, r in zip(axes.flat, records):
    for s, w, lab in r["words"]:
        if not is_content(w): continue
        ax.plot(correlation_curve(r["maps"], s), lw=1.6, label=lab)
    ax.axhline(0.8, color="gray", ls="--", lw=1)
    ax.axvspan(r["maps"].shape[0] - 1.5, r["maps"].shape[0] - 0.5, color="0.9")
    ax.set_title(r["name"], fontsize=9); ax.legend(fontsize=7, frameon=False); ax.set_xlabel("UNet evaluation")
axes[0, 0].set_ylabel("r with final layout"); axes[1, 0].set_ylabel("r with final layout")
for ax in axes.flat[len(records):]: ax.set_axis_off()
plt.tight_layout(); plt.savefig(OUT / "settling_curves.png", dpi=120); plt.show()

[figure saved under Results/m1_word_timelines/]

## 7. Summary

In [8]:
all_drops = [d for r in records for d in r["drops"].values()]
summary = {
    "recorder_vs_stock_mean_pixel_diff": pix,
    "final_step_drop": {"mean": float(np.mean(all_drops)), "min": float(np.min(all_drops)),
                        "max": float(np.max(all_drops)), "n_words": len(all_drops),
                        "note": "r at second-to-last evaluation minus r at last; excluded from commit_step"},
    "prompts": [{"name": r["name"], "prompt": r["prompt"], "seed": r["seed"], "steps": r["steps"],
                 "start_token_share": round(r["sot_share"], 3),
                 "commit_steps": {lab: c for lab, c in r["commits"].items()},
                 "final_step_drop": {k: round(v, 3) for k, v in r["drops"].items()},
                 "threshold_sensitivity": {str(k): v for k, v in sens[r["name"]].items()}}
                for r in records],
}
json.dump(summary, open(OUT / "summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

{
  "recorder_vs_stock_mean_pixel_diff": 0.0,
  "final_step_drop": {
    "mean": 0.09323968264189633,
    "min": 0.0035680532455444336,
    "max": 0.32475966215133667,
    "n_words": 44,
    "note": "r at second-to-last evaluation minus r at last; excluded from commit_step"
  },
  "prompts": [
    {
      "name": "coco_87470",
      "prompt": "A bunch of cows walking on the side of the road",
      "seed": 0,
      "steps": 30,
      "start_token_share": 0.892,
      "commit_steps": {
        "a": 22,
        "bunch": 21,
        "of (1)": 16,
        "cows": 7,
        "walking": 6,
        "on": 4,
        "the (1)": 6,
        "side": 15,
        "of (2)": 4,
        "the (2)": 7,
        "road": 5
      },
      "final_step_drop": {
        "bunch": 0.325,
        "cows": 0.013,
        "walking": 0.014,
        "side": 0.057,
        "road": 0.033
      },
      "threshold_sensitivity": {
        "0.7": {
          "bunch": 19,
          "cows": 5,
          "walking": 4,
        